# Hormuz twin experiment — v4 "spin-up": is the identifiability result a property of the data window?

**Why this notebook exists.** In v3 (sudden start of the boundary forcing) the 4-parameter zonal model was identified from sea-level height alone in 9 of 10 noise realisations at U0 = 1 m/s. In v3f (3-hour smooth start, one-line change) the same row dropped to 0 of 10. Real altimetry sees a quasi-periodic tide, not a start-up transient. Before any more PINN work we must know **what information the h-only data carry once the start-up transient is gone**.

**Hypotheses (declared before running; pre-registration hash is printed by the next cell).**
* **TH-A (transient carries information):** with a cold start (ramp = 0) the linearised uncertainty of the boundary position from h-only data is at least 2x smaller than with the 3-hour ramp, for U0 >= 0.5 m/s.
* **TH-B (a later window does not help):** with the 3-hour ramp, moving the 22.5-hour data window to later times (>= 24 h after the start) never reduces the linearised boundary uncertainty by more than 1.5x relative to the v3f window (2.5-25 h).
* **TH-C (data window and misspecification are separable):** with a truth that belongs to the model family ("smooth" truth, chi2_ref = 1 by construction) the h+u+v advantage over h-only can be judged without the sharp-truth representation error (reported, no pass/fail).

**What runs.**
| block | what | needs | cost (standard) |
|---|---|---|---|
| GATE | NumPy vs JAX forward model; JAX Jacobian vs finite differences | JAX | seconds |
| E1 | how fast does the transient die? periodicity defect of h vs time, steady-time t_ss(U0), period of the leftover oscillation | NumPy only | ~1 min |
| E2 | linearised information (Fisher/Cramer-Rao) at the true parameters for ramp 3 h vs cold start and for data windows starting 2.5-96 h after the start | JAX | ~10 min |
| E3 | classical multi-start identification rows, early window vs late window (same classification as v3/v3f) | JAX | 2-3 h |
| E4 | same rows with a smooth truth (model family) | JAX | ~1 h |
| SUMMARY | tables, pre-registered verdicts, figure, JSON | - | seconds |

**Honest scope.** Idealised channel only. No PINN here (the PINN gates are redesigned separately). E2 is a *linearised* analysis at the true parameters: it says what the data can constrain locally, not where an optimiser ends up. Numbers quoted in the runbook from the authoring sandbox are NumPy finite-difference previews, not results of this notebook.


In [1]:
import os, json, time, hashlib, math, glob
import numpy as np
from scipy.optimize import minimize
from scipy import stats
try:
    import jax, jax.numpy as jnp
    from jax import lax
    jax.config.update("jax_enable_x64", True)
    HAVE_JAX = True
except Exception as _e:
    HAVE_JAX = False
    print("WARNING: JAX not available ->", _e, "\n  GATE (JAX part), E3 and E4 are skipped; E2 falls back to NumPy finite differences.")

MODE = "quick"          # "quick" (pipeline check, ~20-30 min) | "standard"
RUN = dict(GATE=True, E1=True, E2=True, E3=True, E4=True)
RESUME = True
RAMP_HOURS = 3.0           # v3f value; E2 also evaluates ramp = 0 (v3 cold start) explicitly

PREREG = dict(
    version="v4-spinup-prereg-1",
    question="Does the h-only identifiability seen in v3 (cold start) and lost in v3f (3 h ramp) depend on the start-up transient inside the 2.5-25 h data window?",
    window="data window = 54 snapshots, stride 25 steps (rich) or 4 snapshots (sparse4), first snapshot at step W0; W0=150 (2.5 h) reproduces the v3f window exactly",
    E1="periodicity defect d_k = rms(h(t)-h(t-P))/rms(h(t)) over successive blocks of one M2 period P (all cells, truth Cf, ramp 3 h); t_ss(U0) = end of the first block after which d_k < 0.05 for all later blocks; leftover-oscillation period from an FFT of the residual after an M2+overtones harmonic fit (informational)",
    E2="Fisher information of the zonal 4-parameter model at the true parameters (smooth truth), noise sd = 3% of the std of the clean data in the window (as in v3/v3f); report median over layouts of sd(ln Cf_low), sd(ln Cf_high), sd_bnd_km = LX*sqrt(var(b0)+var(s)/12); no absolute pass/fail (the Fisher ellipsoid is stricter than the multi-start classification of v3/v3f), decisions are ratios",
    TH_A="h-only, rich: median sd_bnd(cold start, W0=2.5 h) / sd_bnd(ramp 3 h, W0=2.5 h) <= 0.5 for every U0 in U0_E2 with U0 >= 0.5 -> 'transient carries information'",
    TH_B="h-only, rich, ramp 3 h: for every U0, min over windows W0 >= 24 h of sd_bnd(W0) / sd_bnd(W0=2.5 h) >= 0.67 -> 'no later window is >= 1.5x more informative'",
    E3="classical zonal multi-start rows (arm 'same', sharp truth, ramp 3 h), rich regime, variants h and huv, windows early (W0=150) and late (W0* from E1: first step multiple of 25 at or after t_ss, clipped to [150, W0_CAP]); classification identical to v3/v3f; counts with Clopper-Pearson 95% intervals; 'late better than early' only if the intervals do not overlap",
    E4="same rows as E3 for U0=1 with a smooth truth Cf = cf_zonal(P_TRUE) (model family): reported; expected chi2_ref ~ 1",
    classification="identical to v3f: DCHI2_CRIT 13.3, tolerances 25% (Cf) and 6.25 km (boundary), IDENTIFIED if identified >= ceil(0.9*valid), NOT_IDENTIFIED if <= floor(0.1*valid), else MARGINAL",
    not_in_this_notebook="PINN, indep generator, free-field lambda path, nuisance factors",
)
PREREG_HASH = hashlib.sha256(json.dumps(PREREG, sort_keys=True).encode()).hexdigest()[:16]

_MODES = {
    "quick":    dict(N_STARTS=3, ZONAL_MAXITER=60,  N_SEEDS=2,  U0_E1=[0.5, 1.0], N_LONG_H=96,  U0_E2=[1.0], W0_E2_RAMP_H=[2.5, 24.0], W0_E2_COLD_H=[2.5, 24.0],
                     N_LAYOUTS=2, U0_E3=[1.0], W0_CAP_H=40.0),
    "standard": dict(N_STARTS=6, ZONAL_MAXITER=120, N_SEEDS=10, U0_E1=[0.15, 0.3, 0.5, 1.0], N_LONG_H=240, U0_E2=[0.15, 0.5, 1.0],
                     W0_E2_RAMP_H=[2.5, 12.0, 24.0, 48.0, 96.0], W0_E2_COLD_H=[2.5, 24.0, 96.0], N_LAYOUTS=5, U0_E3=[0.5, 1.0], W0_CAP_H=96.0),
}
CFG = dict(N_STEPS=1500, T_STRIDE=25, WIN_LEN=1350, P_OBS=39, NOISE_FRAC=0.03, SEED=7, CF_SCALE=2.5e-3,
           TOL_BND_KM=6.25, TOL_CF_REL=0.25, DCHI2_CRIT=13.3, SS_THRESH=0.05)
CFG.update(_MODES[MODE])
REGIMES = {}       # filled by set_window()

OUT_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
OUT_PATH = os.path.join(OUT_DIR, "v4_spinup_results.json")
RESULTS = dict(mode=MODE, prereg=PREREG, prereg_hash=PREREG_HASH, have_jax=HAVE_JAX, cfg=CFG, gate={}, E1={}, E2={}, E3={}, E4={})
# resume source: the working dir first, else a previous run attached as an Input dataset (so a session that died can be continued)
_resume_src = [p for p in [OUT_PATH] + sorted(glob.glob("/kaggle/input/**/v4_spinup_results.json", recursive=True)) if os.path.exists(p)]
if RESUME and _resume_src:
    _old = json.load(open(_resume_src[0]))
    if _old.get("prereg_hash") == PREREG_HASH and _old.get("mode") == MODE:
        for k in ("gate", "E1", "E2", "E3", "E4"): RESULTS[k] = _old.get(k, {})
        print("RESUMED from", _resume_src[0])
    else:
        print("WARNING: existing results file", _resume_src[0], "has a different pre-registration hash / mode -> NOT resumed.")

def _jsonable(o):
    if isinstance(o, np.floating): return float(o)
    if isinstance(o, np.integer): return int(o)
    if isinstance(o, np.bool_): return bool(o)
    if isinstance(o, np.ndarray): return o.tolist()
    raise TypeError(type(o))
def save_results():
    with open(OUT_PATH, "w") as f: json.dump(RESULTS, f, default=_jsonable)

print("MODE:", MODE, "| JAX:", HAVE_JAX, "| devices:", (jax.devices() if HAVE_JAX else "n/a"), "| PRE-REGISTRATION HASH:", PREREG_HASH)
print({k: v for k, v in CFG.items()})


MODE: quick | JAX: True | devices: [CudaDevice(id=0), CudaDevice(id=1)] | PRE-REGISTRATION HASH: 0620b759b6289c73
{'N_STEPS': 1500, 'T_STRIDE': 25, 'WIN_LEN': 1350, 'P_OBS': 39, 'NOISE_FRAC': 0.03, 'SEED': 7, 'CF_SCALE': 0.0025, 'TOL_BND_KM': 6.25, 'TOL_CF_REL': 0.25, 'DCHI2_CRIT': 13.3, 'SS_THRESH': 0.05, 'N_STARTS': 3, 'ZONAL_MAXITER': 60, 'N_SEEDS': 2, 'U0_E1': [0.5, 1.0], 'N_LONG_H': 96, 'U0_E2': [1.0], 'W0_E2_RAMP_H': [2.5, 24.0], 'W0_E2_COLD_H': [2.5, 24.0], 'N_LAYOUTS': 2, 'U0_E3': [1.0], 'W0_CAP_H': 40.0}


## Solver and problem definition (identical to v3f; only `set_ramp()` added)

In [2]:
# ===== solver + problem definition: IDENTICAL to v3f cell 3 (only set_ramp() added and JAX parts guarded) =====
# ---- core physics + problem definition (xp-generic: numpy for truth/tests, jax.numpy for inversion) ----
LX, LY = 150e3, 50e3
NX, NY = 24, 12
DX, DY = LX / NX, LY / NY
G, H0, F0, DT = 9.81, 80.0, 6.5e-5, 60.0
OMEGA = 2 * np.pi / (12.42 * 3600.0)
RAMP_TR = RAMP_HOURS * 3600.0
def set_ramp(h):
    """change the smooth-start duration (0 = v3 cold start)."""
    global RAMP_TR, RAMP_HOURS
    RAMP_HOURS = h; RAMP_TR = h * 3600.0
def _ramp(xp, t):
    """smooth start of the boundary forcing: 0.5*(1-cos(pi t/TR)) for t < TR, then 1 (C1-continuous). RAMP_HOURS = 0 -> v3 behaviour."""
    if RAMP_TR <= 0: return 1.0
    return xp.where(t < RAMP_TR, 0.5 * (1.0 - xp.cos(xp.pi * t / RAMP_TR)), 1.0)
CF_LOW_TRUE, CF_HIGH_TRUE = 1.5e-3, 4.0e-3
B0_TRUE, S_TRUE = 0.5, 0.1          # boundary x/LX = B0 + S*(y/LY - 0.5)  (identical to the phase-0 twin)
XC = (np.arange(NX) + 0.5) * DX / LX     # normalized cell-centre coordinates
YC = (np.arange(NY) + 0.5) * DY / LY

def swe_step(xp, state, t, Cf, U0):
    eta, u, v = state
    vp = xp.pad(v, ((1, 1), (0, 0)), mode="edge"); vc = 0.5 * (vp[:, :-1] + vp[:, 1:]); v_at_u = 0.5 * (vc[:-1, :] + vc[1:, :])
    uc = 0.5 * (u[:-1, :] + u[1:, :]); up = xp.pad(uc, ((0, 0), (1, 1)), mode="edge"); u_at_v = 0.5 * (up[:, :-1] + up[:, 1:])
    Cf_u = Cf[np.clip(np.arange(NX + 1), 0, NX - 1), :]
    Cf_v = Cf[:, np.clip(np.arange(NY + 1), 0, NY - 1)]
    su = xp.sqrt(u ** 2 + v_at_u ** 2 + 1e-8); sv = xp.sqrt(u_at_v ** 2 + v ** 2 + 1e-8)
    de_dx = (eta[1:, :] - eta[:-1, :]) / DX; de_dy = (eta[:, 1:] - eta[:, :-1]) / DY
    fu = Cf_u / H0 * u * su; fv = Cf_v / H0 * v * sv
    un = u[1:-1, :] + DT * (-G * de_dx + F0 * v_at_u[1:-1, :] - fu[1:-1, :])
    u_new = xp.concatenate([xp.full((1, NY), U0 * _ramp(xp, t) * xp.cos(OMEGA * t)), un, xp.zeros((1, NY))], axis=0)
    vn = v[:, 1:-1] + DT * (-G * de_dy - F0 * u_at_v[:, 1:-1] - fv[:, 1:-1])
    v_new = xp.concatenate([xp.zeros((NX, 1)), vn, xp.zeros((NX, 1))], axis=1)
    div = (u_new[1:, :] - u_new[:-1, :]) / DX + (v_new[:, 1:] - v_new[:, :-1]) / DY
    return (eta - DT * H0 * div, u_new, v_new)

def run_obs_np(Cf, U0, n_steps, ox, oy):
    """numpy reference run -> (h,u,v) at observed cells for every step: 3 arrays (n_steps, P)."""
    s = (np.zeros((NX, NY)), np.zeros((NX + 1, NY)), np.zeros((NX, NY + 1)))
    H, U, V = [], [], []
    for n in range(n_steps):
        s = swe_step(np, s, n * DT, Cf, U0)
        e, u, v = s
        H.append(e[ox, oy]); U.append((0.5 * (u[:-1] + u[1:]))[ox, oy]); V.append((0.5 * (v[:, :-1] + v[:, 1:]))[ox, oy])
    return np.array(H), np.array(U), np.array(V)

def cf_zonal(xp, p, w_cells=0.5):
    """p = [ln Cf_low, ln Cf_high, b0, s]; smooth two-zone field on the cell grid (width = w_cells cells)."""
    X, Y = np.meshgrid(XC, YC, indexing="ij")
    arg = (X - (p[2] + p[3] * (Y - 0.5))) / (w_cells * DX / LX)
    sig = 1.0 / (1.0 + xp.exp(-arg))
    lo, hi = xp.exp(p[0]), xp.exp(p[1])
    return lo + (hi - lo) * sig

def cf_true_field():
    X, Y = np.meshgrid(XC, YC, indexing="ij")
    return np.where(X < B0_TRUE + S_TRUE * (Y - 0.5), CF_LOW_TRUE, CF_HIGH_TRUE)


# ---- K-zone model: K strips along x separated by K-1 parallel (common-slope) smooth boundaries ----
def cf_zonal_K(xp, theta, K, w_cells=0.5):
    """theta = [ln c_1..ln c_K, b_1..b_{K-1}, s]  (K=1: [ln c_1]).  Boundary k: x/LX = b_k + s*(y/LY-0.5)."""
    X, Y = np.meshgrid(XC, YC, indexing="ij")
    f = xp.exp(theta[0]) * xp.ones((NX, NY))
    for k in range(K - 1):
        arg = (X - (theta[K + k] + theta[2 * K - 1] * (Y - 0.5))) / (w_cells * DX / LX)
        f = f + (xp.exp(theta[k + 1]) - xp.exp(theta[k])) / (1.0 + xp.exp(-arg))
    return f


CF_T = cf_true_field()
BOUNDS = [(np.log(5e-4), np.log(2e-2))] * 2 + [(0.1, 0.9), (-0.5, 0.5)]
P_TRUE = np.array([np.log(CF_LOW_TRUE), np.log(CF_HIGH_TRUE), B0_TRUE, S_TRUE])
if HAVE_JAX:
    # ---- JAX forward model returning only the observed cells (keeps memory tiny) ----
    def make_forward(U0, ox, oy):
        oxj, oyj = jnp.array(ox), jnp.array(oy)
        def fwd(Cf):
            def body(state, n):
                new = swe_step(jnp, state, n * DT, Cf, U0)
                e, u, v = new
                uc = 0.5 * (u[:-1, :] + u[1:, :]); vc = 0.5 * (v[:, :-1] + v[:, 1:])
                return new, (e[oxj, oyj], uc[oxj, oyj], vc[oxj, oyj])
            init = (jnp.zeros((NX, NY)), jnp.zeros((NX + 1, NY)), jnp.zeros((NX, NY + 1)))
            _, ys = lax.scan(body, init, jnp.arange(CFG["N_STEPS"]))
            return ys
        return fwd


    def _build_problem_same(U0, regime, variant, P_use, noise_seed=0, cf_truth=None):
        """ARM 'same': truth from the NumPy loop of the SAME scheme as the inversion solver ('inverse crime'). Layout and noise change with noise_seed."""
        cf_truth = CF_T if cf_truth is None else cf_truth
        rng = np.random.default_rng(CFG["SEED"] + noise_seed)
        cells = rng.choice(NX * NY, CFG["P_OBS"], replace=False)
        ox, oy = np.unravel_index(cells[:P_use], (NX, NY))
        t_idx = REGIMES[regime]
        Hc, Uc, Vc = run_obs_np(cf_truth, U0, CFG["N_STEPS"], ox, oy)
        clean = dict(h=Hc[t_idx], u=Uc[t_idx], v=Vc[t_idx])
        sd = {k: CFG["NOISE_FRAC"] * clean[k].std() for k in clean}
        nr = np.random.default_rng(11 + noise_seed)
        obs = {k: clean[k] + nr.normal(scale=sd[k], size=clean[k].shape) for k in clean}
        fwd = make_forward(U0, ox, oy)
        tj = jnp.array(t_idx)
        def chi2(Cf):
            H, U, V = fwd(Cf)
            c = jnp.mean(((H[tj] - obs["h"]) / sd["h"]) ** 2)
            if variant == "huv":
                c = (c + jnp.mean(((U[tj] - obs["u"]) / sd["u"]) ** 2) + jnp.mean(((V[tj] - obs["v"]) / sd["v"]) ** 2)) / 3.0
            return c
        return dict(U0=U0, regime=regime, variant=variant, P=P_use, ox=ox, oy=oy, t_idx=t_idx, obs=obs, sd=sd,
                    fwd=fwd, chi2=chi2, n_obs=int(obs["h"].size * (3 if variant == "huv" else 1)))

## Tools: windows, E1 decay analysis, E2 Fisher analysis, classical rows

In [3]:
# ===================== new tools for v4 =====================
def regime_idx(W0, regime):
    """snapshot steps of a data window whose first snapshot is at step W0 (W0=150 == v3f window)."""
    if regime == "rich":    return np.arange(W0, W0 + CFG["WIN_LEN"], CFG["T_STRIDE"])
    if regime == "sparse4": return W0 + np.array([225, 599, 974, 1349])
    raise ValueError(regime)

def set_window(W0):
    """set N_STEPS and REGIMES for a data window starting at step W0 (used by the v3f problem builders, which read these globals)."""
    CFG["N_STEPS"] = int(W0 + CFG["WIN_LEN"])
    REGIMES["rich"] = regime_idx(W0, "rich"); REGIMES["sparse4"] = regime_idx(W0, "sparse4")
set_window(150)

def cp_interval(k, n, alpha=0.05):
    """Clopper-Pearson interval for k successes out of n."""
    lo = 0.0 if k == 0 else stats.beta.ppf(alpha / 2, k, n - k + 1)
    hi = 1.0 if k == n else stats.beta.ppf(1 - alpha / 2, k + 1, n - k)
    return float(lo), float(hi)

# ---------------- E1: transient decay (NumPy only) ----------------
P_STEPS = 2 * np.pi / OMEGA / DT           # M2 period in steps (non-integer)

def run_full_np(Cf, U0, n):
    s = (np.zeros((NX, NY)), np.zeros((NX + 1, NY)), np.zeros((NX, NY + 1)))
    H = np.empty((n, NX, NY))
    for k in range(n):
        s = swe_step(np, s, k * DT, Cf, U0); H[k] = s[0]
    return H

def periodicity_defect(H):
    """list of (block end time [h], d_k) for blocks k = 1, 2, ... of one M2 period (block k compared with the same phase one period earlier)."""
    out, k = [], 1
    while (k + 1) * P_STEPS <= len(H):
        idx = np.arange(int(k * P_STEPS), int((k + 1) * P_STEPS))
        prev = idx - P_STEPS; lo = np.floor(prev).astype(int); fr = prev - lo
        hp = H[lo] * (1 - fr)[:, None, None] + H[lo + 1] * fr[:, None, None]
        d = np.sqrt(np.mean((H[idx] - hp) ** 2)) / np.sqrt(np.mean(H[idx] ** 2))
        out.append(((k + 1) * P_STEPS * DT / 3600.0, float(d))); k += 1
    return out

def steady_time(blocks, thr):
    """end time [h] of the first block after which every block has d < thr; None if never."""
    for i in range(len(blocks)):
        if all(b[1] < thr for b in blocks[i:]): return blocks[i][0]
    return None

def residual_period(H, cell=None, n_harm=4):
    """dominant period [h] and relative std of the residual of one mid-channel cell over t in [P, 3P] after fitting const + trend + M2 and its overtones."""
    cell = (NX // 2, NY // 2) if cell is None else cell
    a, b = int(P_STEPS), min(int(3 * P_STEPS), len(H))
    t = np.arange(a, b) * DT; y = H[a:b, cell[0], cell[1]]
    cols = [np.ones_like(t), t - t.mean()]
    for m in range(1, n_harm + 1): cols += [np.cos(m * OMEGA * t), np.sin(m * OMEGA * t)]
    A = np.stack(cols, 1); coef = np.linalg.lstsq(A, y, rcond=None)[0]; r = y - A @ coef
    rw = r * np.hanning(len(r)); nfft = 16 * len(r)
    sp = np.abs(np.fft.rfft(rw, nfft)); f = np.fft.rfftfreq(nfft, DT)
    i = int(np.argmax(sp[1:])) + 1
    return float(1.0 / f[i] / 3600.0), float(np.std(r) / np.std(y))

# ---------------- E2: linearised information ----------------
def layout_cells(layout):
    rng = np.random.default_rng(CFG["SEED"] + layout)
    cells = rng.choice(NX * NY, CFG["P_OBS"], replace=False)
    return np.unravel_index(cells, (NX, NY))

def jac_fd(U0, ox, oy, n, h=1e-4):
    """NumPy central finite differences of the observed (h,u,v) series w.r.t. the 4 zonal parameters at P_TRUE (smooth truth)."""
    base = run_obs_np(cf_zonal(np, P_TRUE), U0, n, ox, oy); cols = []
    for i in range(4):
        pp = P_TRUE.copy(); pp[i] += h; pm = P_TRUE.copy(); pm[i] -= h
        a = run_obs_np(cf_zonal(np, pp), U0, n, ox, oy); b = run_obs_np(cf_zonal(np, pm), U0, n, ox, oy)
        cols.append([(x - y) / (2 * h) for x, y in zip(a, b)])
    return list(base), [np.stack([cols[i][v] for i in range(4)], -1) for v in range(3)]

if HAVE_JAX:
    def jac_jax(U0, ox, oy, n):
        old = CFG["N_STEPS"]; CFG["N_STEPS"] = n
        try:
            fwd = make_forward(U0, ox, oy)
            g = lambda p: fwd(cf_zonal(jnp, p))
            both = jax.jit(lambda p: (g(p), jax.jacfwd(g)(p)))
            base, J = both(jnp.array(P_TRUE))
        finally:
            CFG["N_STEPS"] = old
        return [np.asarray(b) for b in base], [np.asarray(j) for j in J]

def fisher_metrics(base, J, t_idx, variant):
    """linearised uncertainty of (ln Cf_low, ln Cf_high, boundary offset b0, tilt s) for the window t_idx. h-only: all P_OBS cells; huv: first P_OBS//3 cells, h+u+v."""
    Pu = CFG["P_OBS"] if variant == "h" else CFG["P_OBS"] // 3
    vars_ = [0] if variant == "h" else [0, 1, 2]
    rows = []
    for vi in vars_:
        sd = CFG["NOISE_FRAC"] * base[vi][t_idx][:, :Pu].std()
        rows.append(J[vi][t_idx][:, :Pu, :].reshape(-1, 4) / sd)
    Jt = np.vstack(rows); F = Jt.T @ Jt
    sv = np.linalg.svd(Jt, compute_uv=False)
    C = np.linalg.pinv(F); sg = np.sqrt(np.clip(np.diag(C), 0, None))
    return dict(sd_lnClow=float(sg[0]), sd_lnChigh=float(sg[1]), sd_b0_km=float(sg[2] * LX / 1e3), sd_tilt_km=float(sg[3] * LX / 1e3),
                sd_bnd_km=float(LX / 1e3 * math.sqrt(C[2, 2] + C[3, 3] / 12.0)), sv=[float(x) for x in sv], cond_F=float(np.linalg.cond(F)))

# ---------------- E3/E4: classical rows (v3f machinery, arm 'same') ----------------
if HAVE_JAX:
    build_problem = _build_problem_same
    def fit_zonal(pb, p0):
        vgf = jax.jit(jax.value_and_grad(lambda p: pb["chi2"](cf_zonal(jnp, p))))
        def f(p):
            v, g = vgf(jnp.array(p)); return float(v), np.asarray(g, dtype=np.float64)
        res = minimize(f, p0, jac=True, method="L-BFGS-B", bounds=BOUNDS, options=dict(maxiter=CFG["ZONAL_MAXITER"], ftol=1e-12, gtol=1e-8))
        p = res.x
        off = float((p[2] - B0_TRUE) * LX / 1e3); tilt = float((p[3] - S_TRUE) * LX / 1e3)
        return dict(cf_low=float(np.exp(p[0])), cf_high=float(np.exp(p[1])), b0=float(p[2]), s=float(p[3]), chi2=float(res.fun), nit=int(res.nit),
                    bnd_km=float(np.mean(np.abs((p[2] - B0_TRUE) + (p[3] - S_TRUE) * (YC - 0.5))) * LX / 1e3),
                    off_km=off, tilt_km=tilt, edge_km=float(max(abs(off - tilt / 2), abs(off + tilt / 2))))

    def zonal_one_seed(U0, regime, variant, P_use, ns, cf_truth=None):
        pbz = build_problem(U0, regime, variant, P_use, noise_seed=ns, cf_truth=cf_truth)
        chi2_ref = float(pbz["chi2"](cf_zonal(jnp, jnp.array(P_TRUE))))
        sols = []
        for k in range(CFG["N_STARTS"]):
            r = np.random.default_rng(100 + k + 17 * ns)
            p0 = np.array([np.log(2.5e-3) + r.normal(0, 0.4), np.log(2.5e-3) + r.normal(0, 0.4), r.uniform(0.3, 0.7), r.uniform(-0.2, 0.2)])
            sols.append(fit_zonal(pbz, p0))
        return dict(noise_seed=ns, chi2_ref=chi2_ref, n_obs=pbz["n_obs"], sols=sols)

def classify_seed(sd, tol_cf=0.25, tol_bnd=6.25, use_edge=False):
    n = sd["n_obs"]; sols = sd["sols"]; best = min(s["chi2"] for s in sols)
    if best > sd["chi2_ref"] + 2 * np.sqrt(2 / n): return "optimiser_failed"
    equiv = [s for s in sols if n * (s["chi2"] - best) <= CFG["DCHI2_CRIT"]]
    key = "edge_km" if use_edge else "bnd_km"
    ok = all(s[key] <= tol_bnd and abs(s["cf_low"] / CF_LOW_TRUE - 1) <= tol_cf and abs(s["cf_high"] / CF_HIGH_TRUE - 1) <= tol_cf for s in equiv)
    return "identified" if ok else "not_identified"

def misspec_flag(sd): return sd["chi2_ref"] > 1 + 5 * np.sqrt(2 / sd["n_obs"])

def classify_row(per_seed, tol_cf=0.25, tol_bnd=6.25, use_edge=False):
    labs = [classify_seed(sd, tol_cf, tol_bnd, use_edge) for sd in per_seed]
    ni, nn, nf = labs.count("identified"), labs.count("not_identified"), labs.count("optimiser_failed")
    valid = ni + nn
    if valid < 0.8 * len(per_seed) or valid == 0: cls = "INVALID"
    elif ni >= math.ceil(0.9 * valid): cls = "IDENTIFIED"
    elif ni <= math.floor(0.1 * valid): cls = "NOT_IDENTIFIED"
    else: cls = "MARGINAL"
    return dict(identified=ni, not_identified=nn, optimiser_failed=nf, n_seeds=len(per_seed), cls=cls)

def late_W0(U0):
    """first window start (steps, multiple of the stride) at or after the steady time t_ss(U0) from E1, clipped to [150, W0_CAP]."""
    cap = int(CFG["W0_CAP_H"] * 3600 / DT)
    e1 = RESULTS["E1"].get(str(U0), {})
    t_ss = e1.get("t_ss_h")
    flag = "ok"
    if t_ss is None:
        W0 = cap; flag = "steady state NOT reached in E1 run -> cap"
    else:
        W0 = int(math.ceil(t_ss * 3600 / DT / CFG["T_STRIDE"]) * CFG["T_STRIDE"])
        if W0 > cap: W0 = cap; flag = "t_ss beyond cap -> cap"
    return max(150, W0), flag

def run_row(store, tag, U0, variant, n_seeds, W0, cf_truth=None, regime="rich"):
    """run (or resume) one classical row for the window starting at step W0; results stored in store[tag]."""
    set_window(W0)
    P_use = CFG["P_OBS"] if variant == "h" else CFG["P_OBS"] // 3
    rows = store.setdefault(tag, dict(U0=U0, variant=variant, regime=regime, W0=int(W0), seeds=[]))["seeds"]
    t0 = time.time()
    for ns in range(len(rows), n_seeds):
        rows.append(zonal_one_seed(U0, regime, variant, P_use, ns, cf_truth=cf_truth)); save_results()
    c = classify_row(rows); lo, hi = cp_interval(c["identified"], c["n_seeds"])
    mc = float(np.mean([s["chi2_ref"] for s in rows]))
    print(f"[row] {tag:46s} W0={W0*DT/3600:6.1f} h  identified {c['identified']:2d}/{c['n_seeds']}  CP95=[{lo:.2f},{hi:.2f}]  -> {c['cls']:15s} chi2_ref(mean)={mc:5.2f}  flagged={sum(misspec_flag(s) for s in rows)}  [{time.time()-t0:.0f}s]")
    set_window(150)
    return c


## GATE
NumPy vs JAX forward model, JAX Jacobian vs finite differences. If it fails, nothing later may be interpreted.

In [4]:
# ===================== GATE =====================
t0 = time.time(); gate = {}
set_ramp(RAMP_HOURS)
if RUN["GATE"]:
    H = run_full_np(CF_T, 1.0, 3000)
    gate["numpy_long_run_finite"] = bool(np.isfinite(H).all() and np.abs(H).max() < 20.0)
    print(f"numpy long run finite and |h|max = {np.abs(H).max():.3f} m  -> {gate['numpy_long_run_finite']}")
    if HAVE_JAX:
        ox, oy = layout_cells(0); n = 300; errs = {}
        for rh in (3.0, 0.0):
            set_ramp(rh); old = CFG["N_STEPS"]; CFG["N_STEPS"] = n
            fwd = make_forward(1.0, ox, oy)
            yj = [np.asarray(a) for a in jax.jit(lambda: fwd(jnp.array(CF_T)))()]
            CFG["N_STEPS"] = old
            yn = run_obs_np(CF_T, 1.0, n, ox, oy)
            errs[rh] = max(float(np.max(np.abs(a - b)) / (np.std(b) + 1e-30)) for a, b in zip(yj, yn))
        set_ramp(RAMP_HOURS)
        gate["jax_vs_numpy_forward_max_rel"] = errs
        print("JAX vs NumPy forward model, max |diff|/std (ramp 3 h, ramp 0):", errs)
        nJ = 1500; ox, oy = layout_cells(0)
        bj, Jj = jac_jax(1.0, ox, oy, nJ); bf, Jf = jac_fd(1.0, ox, oy, nJ)
        jerr = [float(np.max(np.abs(a - b)) / (np.max(np.abs(b)) + 1e-30)) for a, b in zip(Jj, Jf)]
        gate["jacobian_jax_vs_fd_rel"] = jerr
        print("Jacobian JAX (jacfwd) vs NumPy central differences, max rel diff for h, u, v:", [f"{x:.2e}" for x in jerr])
        gate["passed"] = bool(gate["numpy_long_run_finite"] and all(e < 1e-8 for e in errs.values()) and all(e < 1e-3 for e in jerr))
    else:
        gate["passed"] = False
        print("JAX unavailable: only the NumPy check ran; E3/E4 will be skipped.")
    RESULTS["gate"] = gate; save_results()
    print("GATE", "PASSED" if gate["passed"] else "FAILED", f"[{time.time()-t0:.0f}s]")
    if HAVE_JAX and not gate["passed"]:
        raise RuntimeError("GATE FAILED - do not interpret any later result; send this output to Claude.")


numpy long run finite and |h|max = 6.782 m  -> True
JAX vs NumPy forward model, max |diff|/std (ramp 3 h, ramp 0): {3.0: 2.067331594027341e-13, 0.0: 1.2618516104024743e-13}
Jacobian JAX (jacfwd) vs NumPy central differences, max rel diff for h, u, v: ['2.44e-07', '5.37e-07', '1.69e-06']
GATE PASSED [7s]


## E1 — how fast does the start-up transient die?
NumPy only. Periodicity defect = how different the tide is from itself one M2 period earlier.

In [5]:
# ===================== E1: how fast does the start-up transient die? (NumPy only) =====================
if RUN["E1"]:
    set_ramp(RAMP_HOURS)
    n = int(CFG["N_LONG_H"] * 3600 / DT)
    c_seiche = math.sqrt(G * H0)
    print(f"reference scales: M2 period P = {2*math.pi/OMEGA/3600:.2f} h | free gravity-wave (seiche) periods 2L/(n c): n=1 {2*LX/c_seiche/3600:.2f} h, n=2 {LX/c_seiche/3600:.2f} h | steady threshold d < {CFG['SS_THRESH']}")
    for U0 in CFG["U0_E1"]:
        key = str(U0)
        if key not in RESULTS["E1"]:
            t0 = time.time()
            Hf = run_full_np(CF_T, U0, n)
            bl = periodicity_defect(Hf); tss = steady_time(bl, CFG["SS_THRESH"]); per, rrel = residual_period(Hf)
            tau_est = 1.0 / (8 / (3 * math.pi) * 2.75e-3 * U0 / H0) / 3600.0
            RESULTS["E1"][key] = dict(blocks=bl, t_ss_h=tss, resid_period_h=per, resid_rel=rrel, tau_est_h=tau_est, run_s=time.time() - t0)
            save_results()
        r = RESULTS["E1"][key]
        d25 = min(r["blocks"], key=lambda b: abs(b[0] - 25.0))[1]
        print(f"U0={U0:5.2f}  defect at ~25 h = {d25:.3f}  t_ss = {r['t_ss_h'] if r['t_ss_h'] is None else round(r['t_ss_h'],1)} h  "
              f"(order-of-magnitude damping time ~{r['tau_est_h']:.0f} h)  leftover oscillation: period {r['resid_period_h']:.2f} h, size {100*r['resid_rel']:.1f}% of the tide signal")
    print("blocks (end time h : defect):")
    for U0 in CFG["U0_E1"]:
        bl = RESULTS["E1"][str(U0)]["blocks"]
        print(f"  U0={U0:5.2f}: " + "  ".join(f"{b[0]:.0f}h:{b[1]:.3f}" for b in bl[::2]))


reference scales: M2 period P = 12.42 h | free gravity-wave (seiche) periods 2L/(n c): n=1 2.97 h, n=2 1.49 h | steady threshold d < 0.05
U0= 0.50  defect at ~25 h = 0.421  t_ss = None h  (order-of-magnitude damping time ~19 h)  leftover oscillation: period 1.49 h, size 16.2% of the tide signal
U0= 1.00  defect at ~25 h = 0.354  t_ss = 86.9 h  (order-of-magnitude damping time ~10 h)  leftover oscillation: period 1.49 h, size 10.7% of the tide signal
blocks (end time h : defect):
  U0= 0.50: 25h:0.421  50h:0.261  75h:0.169
  U0= 1.00: 25h:0.354  50h:0.142  75h:0.065


## E2 — linearised information vs data window and start type
Fisher analysis at the true parameters; compares cold start (v3) with 3 h ramp (v3f) and early with late windows.

In [6]:
# ===================== E2: linearised information vs data window and start type =====================
if RUN["E2"]:
    E2 = RESULTS["E2"]; t0 = time.time()
    cases = [("ramp3", 3.0, CFG["W0_E2_RAMP_H"]), ("cold", 0.0, CFG["W0_E2_COLD_H"])]
    jac = jac_jax if HAVE_JAX else jac_fd
    print("Jacobian engine:", "JAX jacfwd" if HAVE_JAX else "NumPy finite differences (JAX unavailable)")
    for tag, rh, W0h in cases:
        set_ramp(rh)
        for U0 in CFG["U0_E2"]:
            key = f"{tag}|U0={U0}"
            if key in E2: continue
            W0s = [int(round(h * 3600 / DT)) for h in W0h]; n = max(W0s) + CFG["WIN_LEN"]; out = {}
            for layout in range(CFG["N_LAYOUTS"]):
                ox, oy = layout_cells(layout)
                base, J = jac(U0, ox, oy, n)
                for W0, h in zip(W0s, W0h):
                    for variant in ("h", "huv"):
                        for regime in ("rich", "sparse4"):
                            out.setdefault(f"W0={h}h|{variant}|{regime}", []).append(fisher_metrics(base, J, regime_idx(W0, regime), variant))
            E2[key] = out; save_results()
            print(f"  done {key}  [{time.time()-t0:.0f}s]")
    set_ramp(RAMP_HOURS)


Jacobian engine: JAX jacfwd
  done ramp3|U0=1.0  [3s]
  done cold|U0=1.0  [5s]


## E3 — classical identification rows: early window vs late window
Same classification as v3/v3f. The early window reproduces v3f (reproduction check).

In [7]:
# ===================== E3: classical identification rows, early vs late window =====================
E3 = RESULTS["E3"]
if RUN["E3"] and HAVE_JAX and RESULTS["gate"].get("passed"):
    set_ramp(RAMP_HOURS)
    for U0 in CFG["U0_E3"]:
        W0L, flag = late_W0(U0)
        E3.setdefault("_windows", {})[str(U0)] = dict(W0_late=int(W0L), late_start_h=W0L * DT / 3600.0, flag=flag)
        print(f"U0={U0}: late window starts at step {W0L} ({W0L*DT/3600:.1f} h)  [{flag}]")
        for variant in ("h", "huv"):
            run_row(E3, f"U0={U0}|{variant}|early", U0, variant, CFG["N_SEEDS"], 150)
            run_row(E3, f"U0={U0}|{variant}|late",  U0, variant, CFG["N_SEEDS"], W0L)
    save_results()
    # reproduction check against the v3f 'same' arm (early window = v3f window; identical seeds and settings)
    cands = glob.glob("/kaggle/input/**/v3f_results*.json", recursive=True) + glob.glob("./v3f_results*.json")
    if cands:
        d = json.load(open(cands[0])); same = d["arms"]["same"]["zonal_rows"]
        print("reproduction check vs", cands[0], "(counts identified / seeds; v3f used", d["cfg"]["N_SEEDS"], "seeds):")
        for tag, row in E3.items():
            if tag.startswith("_") or not tag.endswith("|early"): continue
            k = f"U0={row['U0']}|rich|{row['variant']}"
            if k in same:
                a = classify_row(row["seeds"]); b = classify_row(same[k][:len(row["seeds"])])
                print(f"   {tag:24s} now {a['identified']}/{a['n_seeds']}  v3f {b['identified']}/{b['n_seeds']}  {'MATCH' if a['identified']==b['identified'] else 'DIFFERENT (GPU non-determinism or code drift: report)'}")
    else:
        print("v3f_results.json not found in /kaggle/input or the working dir -> reproduction check skipped.")
else:
    print("E3 skipped (RUN flag off, JAX unavailable, or gate not passed).")


U0=1.0: late window starts at step 2400 (40.0 h)  [t_ss beyond cap -> cap]
[row] U0=1.0|h|early                                 W0=   2.5 h  identified  2/2  CP95=[0.16,1.00]  -> IDENTIFIED      chi2_ref(mean)= 1.00  flagged=0  [44s]
[row] U0=1.0|h|late                                  W0=  40.0 h  identified  1/2  CP95=[0.01,0.99]  -> MARGINAL        chi2_ref(mean)= 1.00  flagged=0  [88s]
[row] U0=1.0|huv|early                               W0=   2.5 h  identified  1/2  CP95=[0.01,0.99]  -> INVALID         chi2_ref(mean)= 1.05  flagged=0  [37s]
[row] U0=1.0|huv|late                                W0=  40.0 h  identified  2/2  CP95=[0.16,1.00]  -> IDENTIFIED      chi2_ref(mean)= 1.06  flagged=0  [89s]
reproduction check vs /kaggle/input/datasets/saeedhadian/v3f-results-json/v3f_results.json (counts identified / seeds; v3f used 10 seeds):
   U0=1.0|h|early           now 2/2  v3f 0/2  DIFFERENT (GPU non-determinism or code drift: report)
   U0=1.0|huv|early         now 1/2  v3f 2/2  DIFF

## E4 — smooth truth control
Truth from the model family: separates 'information' from 'representation mismatch'.

In [8]:
# ===================== E4: smooth truth (model family) control =====================
E4 = RESULTS["E4"]
if RUN["E4"] and HAVE_JAX and RESULTS["gate"].get("passed"):
    set_ramp(RAMP_HOURS)
    cf_smooth = np.array(cf_zonal(np, P_TRUE))
    U0 = 1.0
    if U0 in CFG["U0_E3"]:
        W0L = RESULTS["E3"].get("_windows", {}).get(str(U0), {}).get("W0_late") or late_W0(U0)[0]
        for variant in ("h", "huv"):
            run_row(E4, f"U0={U0}|{variant}|early|smooth", U0, variant, CFG["N_SEEDS"], 150, cf_truth=cf_smooth)
            run_row(E4, f"U0={U0}|{variant}|late|smooth",  U0, variant, CFG["N_SEEDS"], W0L, cf_truth=cf_smooth)
        save_results()
else:
    print("E4 skipped (RUN flag off, JAX unavailable, or gate not passed).")


[row] U0=1.0|h|early|smooth                          W0=   2.5 h  identified  2/2  CP95=[0.16,1.00]  -> IDENTIFIED      chi2_ref(mean)= 1.00  flagged=0  [43s]
[row] U0=1.0|h|late|smooth                           W0=  40.0 h  identified  0/2  CP95=[0.00,0.84]  -> NOT_IDENTIFIED  chi2_ref(mean)= 1.00  flagged=0  [99s]
[row] U0=1.0|huv|early|smooth                        W0=   2.5 h  identified  1/2  CP95=[0.01,0.99]  -> INVALID         chi2_ref(mean)= 1.00  flagged=0  [36s]
[row] U0=1.0|huv|late|smooth                         W0=  40.0 h  identified  2/2  CP95=[0.16,1.00]  -> IDENTIFIED      chi2_ref(mean)= 1.00  flagged=0  [88s]


## SUMMARY

In [9]:
# ===================== SUMMARY: tables, pre-registered verdicts, figure =====================
def med(entries, field): return float(np.median([e[field] for e in entries]))
print("=" * 100); print("v4 spin-up | prereg hash", PREREG_HASH, "| mode", MODE, "| JAX", HAVE_JAX); print("=" * 100)

# ---- E1 ----
print("\nE1  transient decay (truth Cf, ramp 3 h)")
for U0 in CFG["U0_E1"]:
    r = RESULTS["E1"].get(str(U0))
    if r: print(f"   U0={U0:5.2f}  t_ss = {r['t_ss_h'] if r['t_ss_h'] is None else round(r['t_ss_h'],1)} h   estimate ~{r['tau_est_h']:.0f} h   leftover period {r['resid_period_h']:.2f} h")

# ---- E2 ----
print("\nE2  linearised uncertainty (median over layouts).  sd_lnC ~ relative error of Cf; sd_bnd = rms boundary-position error [km].  ratio = sd_bnd / sd_bnd(ramp 3 h, window 2.5 h)")
verdict = {}
for U0 in CFG["U0_E2"]:
    ref_key = f"ramp3|U0={U0}"; ref = RESULTS["E2"].get(ref_key, {}).get("W0=2.5h|h|rich")
    if not ref: continue
    ref_b = med(ref, "sd_bnd_km")
    print(f"\n   U0 = {U0}")
    print("   start    window   variant regime  sd_lnClow sd_lnChigh sd_bnd_km  ratio")
    for tag in ("cold", "ramp3"):
        E = RESULTS["E2"].get(f"{tag}|U0={U0}", {})
        for k in sorted(E, key=lambda s: (float(s.split("|")[0][3:-1]), s)):
            w, var, reg = k.split("|")
            if reg == "sparse4" and var == "huv": continue
            m = E[k]
            print(f"   {tag:6s}  {w:9s} {var:6s} {reg:8s} {med(m,'sd_lnClow'):9.3f} {med(m,'sd_lnChigh'):10.3f} {med(m,'sd_bnd_km'):9.2f} {med(m,'sd_bnd_km')/ref_b:7.2f}")
    cold = RESULTS["E2"].get(f"cold|U0={U0}", {}).get("W0=2.5h|h|rich")
    late = [med(v, "sd_bnd_km") for k, v in RESULTS["E2"].get(ref_key, {}).items() if k.endswith("|h|rich") and float(k.split("|")[0][3:-1]) >= 24.0]
    verdict[U0] = dict(A=(med(cold, "sd_bnd_km") / ref_b) if cold else None, B=(min(late) / ref_b) if late else None)
print("\n   NOTE: in sparse4 the noise sd is 3% of the std of only 4 snapshots, so it depends on the tidal phase that is sampled; sparse4 rows must NOT be compared across windows (verdicts use the rich regime only).")
print("\nPRE-REGISTERED VERDICTS (E2)")
A_list = [(U0, v["A"]) for U0, v in verdict.items() if U0 >= 0.5 and v["A"] is not None]
B_list = [(U0, v["B"]) for U0, v in verdict.items() if v["B"] is not None]
if A_list:
    okA = all(x <= 0.5 for _, x in A_list)
    print("   TH-A (cold start >= 2x more informative, U0>=0.5):", "SUPPORTED" if okA else "NOT SUPPORTED", {u: round(x, 2) for u, x in A_list})
else: print("   TH-A: not evaluable (no U0 >= 0.5 with both cases)")
if B_list:
    okB = all(x >= 0.67 for _, x in B_list)
    print("   TH-B (no later window >= 1.5x better):", "SUPPORTED" if okB else "NOT SUPPORTED", {u: round(x, 2) for u, x in B_list})
else: print("   TH-B: not evaluable")
RESULTS["verdict_E2"] = dict(A={str(u): x for u, x in A_list}, B={str(u): x for u, x in B_list})

# ---- E3 / E4 ----
def print_rows(title, store):
    print(f"\n{title}")
    for tag, row in store.items():
        if tag.startswith("_"): continue
        c = classify_row(row["seeds"]); lo, hi = cp_interval(c["identified"], c["n_seeds"])
        print(f"   {tag:36s} W0={row['W0']*DT/3600:6.1f} h  identified {c['identified']:2d}/{c['n_seeds']}  CP95=[{lo:.2f},{hi:.2f}]  {c['cls']:15s} chi2_ref={np.mean([s['chi2_ref'] for s in row['seeds']]):5.2f}")
if RESULTS["E3"]: print_rows("E3  classical rows, sharp truth (v3f truth)", RESULTS["E3"])
if RESULTS["E4"]: print_rows("E4  classical rows, smooth truth (model family)", RESULTS["E4"])
if RESULTS["E3"]:
    print("\nE3 pre-registered reading: 'late better than early' only if the 95% intervals do not overlap")
    for U0 in CFG["U0_E3"]:
        for var in ("h", "huv"):
            a, b = RESULTS["E3"].get(f"U0={U0}|{var}|early"), RESULTS["E3"].get(f"U0={U0}|{var}|late")
            if a and b:
                ca, cb = classify_row(a["seeds"]), classify_row(b["seeds"])
                ia, ib = cp_interval(ca["identified"], ca["n_seeds"]), cp_interval(cb["identified"], cb["n_seeds"])
                res = "late better" if ib[0] > ia[1] else ("early better" if ia[0] > ib[1] else "not distinguishable")
                print(f"   U0={U0} {var:3s}: early {ca['identified']}/{ca['n_seeds']}  late {cb['identified']}/{cb['n_seeds']}  -> {res}")
save_results()

# ---- figure ----
try:
    import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
    fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
    for U0 in CFG["U0_E1"]:
        r = RESULTS["E1"].get(str(U0))
        if r: ax[0].semilogy([b[0] for b in r["blocks"]], [max(b[1], 1e-5) for b in r["blocks"]], "o-", label=f"U0={U0}")
    ax[0].axhline(CFG["SS_THRESH"], color="k", ls="--", lw=0.8); ax[0].axvspan(2.5, 25, color="orange", alpha=0.15, label="v3f data window")
    ax[0].set_xlabel("time [h]"); ax[0].set_ylabel("periodicity defect of h"); ax[0].set_title("E1: how fast the transient dies"); ax[0].legend(fontsize=8)
    for tag, ls in (("cold", "--"), ("ramp3", "-")):
        for U0 in CFG["U0_E2"]:
            E = RESULTS["E2"].get(f"{tag}|U0={U0}", {})
            ks = sorted([k for k in E if k.endswith("|h|rich")], key=lambda s: float(s.split("|")[0][3:-1]))
            if ks: ax[1].semilogy([float(k.split("|")[0][3:-1]) for k in ks], [med(E[k], "sd_bnd_km") for k in ks], "o" + ls, label=f"{tag} U0={U0}")
    ax[1].set_xlabel("window start [h]"); ax[1].set_ylabel("sd of boundary position [km] (h-only, rich)"); ax[1].set_title("E2: linearised information"); ax[1].legend(fontsize=7)
    plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "v4_spinup_summary.png"), dpi=130); print("\nfigure saved:", os.path.join(OUT_DIR, "v4_spinup_summary.png"))
except Exception as e:
    print("figure skipped:", e)


v4 spin-up | prereg hash 0620b759b6289c73 | mode quick | JAX True

E1  transient decay (truth Cf, ramp 3 h)
   U0= 0.50  t_ss = None h   estimate ~19 h   leftover period 1.49 h
   U0= 1.00  t_ss = 86.9 h   estimate ~10 h   leftover period 1.49 h

E2  linearised uncertainty (median over layouts).  sd_lnC ~ relative error of Cf; sd_bnd = rms boundary-position error [km].  ratio = sd_bnd / sd_bnd(ramp 3 h, window 2.5 h)

   U0 = 1.0
   start    window   variant regime  sd_lnClow sd_lnChigh sd_bnd_km  ratio
   cold    W0=2.5h   huv    rich         0.017      0.010      0.30    0.01
   cold    W0=2.5h   h      rich         0.075      0.050      4.43    0.22
   cold    W0=2.5h   h      sparse4      0.052      0.046      3.43    0.17
   cold    W0=24.0h  huv    rich         0.018      0.014      0.40    0.02
   cold    W0=24.0h  h      rich         0.088      0.076      5.22    0.26
   cold    W0=24.0h  h      sparse4      0.758      0.681     32.71    1.60
   ramp3   W0=2.5h   huv    rich   